# BÀI TẬP CHƯƠNG 4 - DEEP LEARNING VỚI KERAS

**Chủ đề:** Classification and Regression  
**Ngôn ngữ:** Python + TensorFlow/Keras  
**Mục tiêu:** Thực hành các bài toán tiêu biểu trong Chương 4:
- Phân loại nhị phân với IMDB
- Phân loại đa lớp nhỏ với Iris
- Phân loại đa lớp với Reuters
- Hồi quy với Boston Housing

---

## Yêu cầu chung
1. Hoàn thành các phần còn thiếu trong các ô code.
2. Chạy từng phần theo đúng thứ tự.
3. Trả lời ngắn gọn các câu hỏi thảo luận ở cuối mỗi bài.
4. Không sửa tên biến nếu không cần thiết.

---

## Chuẩn đầu ra
Sau khi hoàn thành notebook này, sinh viên có thể:
- Nạp và tiền xử lý dữ liệu cho bài toán phân loại và hồi quy.
- Xây dựng mô hình phân loại nhị phân, đa lớp và hồi quy bằng Keras.
- Chọn activation, loss, metric phù hợp với từng bài toán.
- Quan sát validation loss / validation accuracy để phát hiện overfitting.
- Chuẩn hóa đặc trưng số cho các bài toán cần thiết.
- Áp dụng K-fold validation khi dữ liệu ít.


In [ ]:
# Cài đặt / import thư viện cần thiết
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb, reuters, boston_housing

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

print('TensorFlow / Keras / scikit-learn đã sẵn sàng!')


# PHẦN 1 - IMDB: PHÂN LOẠI NHỊ PHÂN

Trong phần này, bạn sẽ xây dựng mô hình phân loại cảm xúc của review phim: **tích cực / tiêu cực**.

## Bài 1.1 - Nạp dữ liệu IMDB

**Yêu cầu:**
- Nạp dữ liệu IMDB với 10,000 từ phổ biến nhất.
- In ra số lượng mẫu train và test.
- In thử độ dài của một sample đầu tiên.

In [ ]:
# TODO: Nạp dữ liệu IMDB với num_words=10000
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

# TODO: In kích thước tập train và test
print(len(train_data), len(test_data))

# TODO: In độ dài sample đầu tiên của train_data
print(len(train_data[0]))


## Bài 1.2 - Vector hóa dữ liệu

**Yêu cầu:**
- Viết hàm `vectorize_sequences()` để biến mỗi review thành vector nhị phân kích thước 10,000.
- Vector hóa `train_data` và `test_data`.
- Chuyển nhãn sang `float32`.

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    # TODO: Tạo ma trận toàn số 0 với shape phù hợp
    def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    # TODO: Duyệt từng sequence và gán 1.0 tại các vị trí từ xuất hiện
    for i, sequence in enumerate(sequences):
    results[i, sequence] = 1.0  # đánh dấu vị trí từ xuất hiện
    return results

# TODO: Vector hóa train_data và test_data
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# TODO: Chuyển train_labels, test_labels sang float32
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

# TODO: In shape của x_train và x_test
print(x_train.shape)
print(x_test.shape)


## Bài 1.3 - Tạo tập validation

**Yêu cầu:**
- Lấy 8,000 mẫu đầu của tập train làm validation.
- Phần còn lại dùng để train.
- Tạo các biến: `x_val`, `partial_x_train`, `y_val`, `partial_y_train`.

In [ ]:
# TODO: Tách validation set và partial train set
x_val = x_train[:8000]
partial_x_train = x_train[8000:]
y_val = y_train[:8000]
partial_y_train = y_train[8000:]
# Gợi ý:
# x_val = ...
# partial_x_train = ...
# y_val = ...
# partial_y_train = ...

# TODO: In shape để kiểm tra
print(x_val.shape, partial_x_train.shape)


## Bài 1.4 - Xây dựng mô hình IMDB

**Yêu cầu:**
- Tạo mô hình `Sequential` gồm:
  - Dense(16, relu)
  - Dense(16, relu)
  - Output layer: tự khai báo cho phù hợp
- Compile với:
  - optimizer = `rmsprop`
  - loss = `binary_crossentropy`
  - metrics = `accuracy`

In [ ]:
# TODO: Tạo mô hình imdb_model
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(10000,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'))]
# TODO: Compile mô hình
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])
# Xem cấu trúc model
model.summary()

## Bài 1.5 - Huấn luyện mô hình và theo dõi validation

**Yêu cầu:**
- Train mô hình trong 20 epochs.
- batch_size = 256.
- Lưu kết quả vào biến `history`.

In [ ]:
# TODO: Train mô hình với validation_data=(x_val, y_val)
history = model.fit(partial_x_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(x_val, y_val))

# history = ...


## Bài 1.6 - Vẽ biểu đồ loss

**Yêu cầu:**
- Lấy `loss` và `val_loss` từ `history.history`.
- Tạo danh sách `epochs`.
- Vẽ 2 đường/chuỗi điểm:
  - Training loss
  - Validation loss
- Quan sát biểu đồ và cho biết: Nên chọn bao nhiêu epochs là hợp lý? Lý do?

In [ ]:
# TODO: Lấy loss và val_loss từ history
loss = history.history['loss']
val_loss = history.history['val_loss']

# TODO: Tạo biến epochs
epochs = range(1, len(loss) + 1)

# TODO: Vẽ biểu đồ training loss và validation loss
plt.plot(epochs, loss, 'bo-', label='Training loss')
plt.plot(epochs, val_loss, 'r^-', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Gợi ý:
# plt.plot(epochs, loss_values, 'bo', label='Training loss')
# plt.plot(epochs, val_loss_values, 'b', label='Validation loss')
# plt.legend()
# plt.show()


## Bài 1.7 - Huấn luyện mô hình cuối cùng

**Yêu cầu:**
- Tạo lại một mô hình mới cùng kiến trúc.
- Huấn luyện với số epoch phù hợp hơn dựa trên biểu đồ validation loss.
- Đánh giá trên test set.

In [ ]:
# TODO: Tạo lại mô hình imdb_final_model
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(10000,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')]

# TODO: Compile mô hình
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# TODO: Train lại mô hình trên toàn bộ x_train và y_train
model.fit(x_train, y_train, epochs=4, batch_size=256)

# TODO: Đánh giá trên x_test và y_test
results = model.evaluate(x_test, y_test)

# TODO: In kết quả test
print("Test Loss:", results[0])
print("Test Accuracy:", results[1])


## Câu hỏi thảo luận - IMDB

1. Vì sao lớp cuối dùng `sigmoid` thay vì `softmax`?
2 lớp sigmoid, nhiều hơn softmax
2. Vì sao bài toán này dùng `binary_crossentropy`?
Nếu dùng categorical_crossentropy thì phải chuyển sang dạng one-hot
3. Dấu hiệu nào trên biểu đồ cho thấy mô hình bắt đầu overfit?
Model học quá kỹ dữ liệu train
4. Nếu tăng số neuron ở các lớp ẩn quá nhiều thì điều gì có thể xảy ra?
Tốn thời gian train, tốn tài nguyên

# PHẦN 2 - IRIS: PHÂN LOẠI ĐA LỚP NHỎ

Trong phần này, bạn sẽ xây dựng mô hình phân loại 3 loài hoa Iris dựa trên 4 đặc trưng số.

## Bài 2.1 - Nạp dữ liệu Iris

**Yêu cầu:**
- Nạp bộ dữ liệu Iris bằng `load_iris()`.
- In shape của dữ liệu và nhãn.
- In tên các lớp.

In [ ]:
# TODO: Nạp bộ dữ liệu Iris
iris = load_iris()

# TODO: Tạo biến data và targets
X = iris.data
y = iris.target

# TODO: In shape của data, targets
print("Shape của X:", X.shape)
print("Shape của y:", y.shape)

# TODO: In tên các lớp
print("Tên các lớp:", iris.target_names)


## Bài 2.2 - Chia train/test và chuẩn hóa dữ liệu

**Yêu cầu:**
- Chia dữ liệu thành train/test theo tỉ lệ 80/20.
- Dùng `random_state=42`.
- Tính mean/std từ train set.
- Chuẩn hóa cả train và test bằng mean/std của train.

In [ ]:
# TODO: Chia train/test cho Iris
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# TODO: Tính mean từ x_train_iris
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# TODO: Chuẩn hóa x_train_iris và x_test_iris
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


# TODO: Kiểm tra mean/std của x_train_iris sau chuẩn hóa
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


## Bài 2.3 - One-hot hóa nhãn Iris

**Yêu cầu:**
- Dùng `keras.utils.to_categorical()` để one-hot hóa nhãn train/test.
- In shape của nhãn sau one-hot.

In [ ]:
# TODO: One-hot hóa y_train_iris và y_test_iris
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# TODO: In shape của nhãn sau one-hot
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)


## Bài 2.4 - Tạo validation set cho Iris

**Yêu cầu:**
- Lấy 20% từ tập train làm validation.
- Tạo các biến:
  - `x_val_iris`, `partial_x_train_iris`
  - `y_val_iris`, `partial_y_train_iris`

In [ ]:
# TODO: Tự tách validation set từ x_train_iris và y_train_iris_cat
val_size = int(0.2 * len(X_train))
x_val_iris = X_train[:val_size]
y_val_iris = y_train[:val_size]
partial_x_train_iris = X_train[val_size:]
partial_y_train_iris = y_train[val_size:]
# TODO: In shape để kiểm tra
print("x_val_iris:", x_val_iris.shape)
print("partial_x_train_iris:", partial_x_train_iris.shape)


## Bài 2.5 - Xây dựng mô hình Iris

**Yêu cầu:**
- Tạo mô hình gồm:
  - Dense(16, relu)
  - Dense(16, relu)
  - Output layer: khai báo cho phù hợp
- Compile với:
  - optimizer = `rmsprop`
  - loss = `categorical_crossentropy`
  - metrics = `accuracy`

In [ ]:
# TODO: Tạo mô hình iris_model
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(4,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax') )]

# TODO: Compile mô hình
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
# TODO: Hiển thị summary
model.summary()

## Bài 2.6 - Huấn luyện và theo dõi validation

**Yêu cầu:**
- Train mô hình 50 epochs.
- batch_size = 16.
- Theo dõi validation accuracy và validation loss.

In [ ]:
# TODO: Train mô hình Iris
history_iris = model.fit(
    partial_x_train_iris,
    partial_y_train_iris,
    epochs=50,
    batch_size=16,
    validation_data=(x_val_iris, y_val_iris))

# history_iris = ...


## Bài 2.7 - Vẽ biểu đồ accuracy của Iris

**Yêu cầu:**
- Lấy `accuracy` và `val_accuracy` từ `history_iris.history`.
- Vẽ biểu đồ so sánh train accuracy và validation accuracy.
- Quan sát biểu đồ và cho biết: Nên chọn bao nhiêu epochs là hợp lý? Lý do?

In [ ]:
# TODO: Trích xuất accuracy và val_accuracy
acc = history_iris.history['accuracy']
val_acc = history_iris.history['val_accuracy']
# TODO: Tạo epochs_iris
epochs = range(1, len(acc) + 1)

# TODO: Vẽ biểu đồ accuracy
epochs = range(1, len(acc) + 1)

# Vẽ biểu đồ
plt.plot(epochs, acc, 'bo-', label='Training accuracy')
plt.plot(epochs, val_acc, 'r^-', label='Validation accuracy')
plt.title('Training and Validation Accuracy (Iris)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Bài 2.8 - Đánh giá trên test set

**Yêu cầu:**
- Tạo lại mô hình Iris mới.
- Train với số epoch hợp lý hơn nếu cần.
- Đánh giá trên test set.

In [ ]:
# TODO: Tạo iris_final_model
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(4,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')])

# TODO: Compile mô hình
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# TODO: Train trên toàn bộ x_train_iris và y_train_iris_cat
model.fit(X_train, y_train, epochs=25, batch_size=16)

# TODO: Đánh giá trên x_test_iris và y_test_iris_cat
results = model.evaluate(X_test, y_test)

# TODO: In test_loss_iris và test_acc_iris
print("Test Loss:", results[0])
print("Test Accuracy:", results[1])


## Câu hỏi thảo luận - Iris

1. Vì sao bài toán Iris là bài toán phân loại đa lớp?
hơn 2 lớp là multi class
2. Vì sao lớp cuối nên dùng `softmax`?
Phù hợp với bài toán phân loại nhiều lớp
3. Vì sao số neuron ở lớp cuối phải là 3?
Vì có 3 lớp cần dự đoán
Mỗi neuron đại diện cho 1 lớp
4. Nếu không chuẩn hóa dữ liệu đầu vào thì mô hình có thể gặp khó khăn gì?
Hội tụ chậm, ko ổn định

# PHẦN 3 - REUTERS: PHÂN LOẠI ĐA LỚP

Trong phần này, bạn sẽ phân loại bản tin Reuters vào nhiều chủ đề khác nhau.

## Bài 3.1 - Nạp dữ liệu Reuters

**Yêu cầu:**
- Nạp dữ liệu Reuters với 10,000 từ phổ biến nhất.
- In số lượng mẫu train và test.
- In số lượng lớp khác nhau.

In [ ]:
# TODO: Nạp dữ liệu Reuters
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

# TODO: In số lượng mẫu train và test
print("Số mẫu train:", len(train_data))
print("Số mẫu test:", len(test_data))

# TODO: In số lượng lớp khác nhau trong train_labels
num_classes = len(np.unique(train_labels))
print("Số lớp:", num_classes)


## Bài 3.2 - Vector hóa dữ liệu và one-hot labels

**Yêu cầu:**
- Dùng lại ý tưởng vector hóa sequence như bài IMDB.
- Viết hàm `to_one_hot()` để one-hot hóa nhãn.
- Tạo `x_train`, `x_test`, `one_hot_train_labels`, `one_hot_test_labels`.

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    # TODO
        results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

def to_one_hot(labels, dimension=46):
    # TODO: Tạo ma trận toàn số 0
    results = np.zeros((len(labels), dimension))
    # TODO: Gán 1.0 tại vị trí lớp đúng
    for i, label in enumerate(labels):
        results[i, label] = 1.0
    return results

# TODO: Vector hóa x_train, x_test
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# TODO: One-hot hóa nhãn train và test
one_hot_train_labels = to_one_hot(train_labels)
one_hot_test_labels = to_one_hot(test_labels)

# TODO: In shape để kiểm tra
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("one_hot_train_labels:", one_hot_train_labels.shape)


## Bài 3.3 - Tạo validation set cho Reuters

**Yêu cầu:**
- Lấy 1,000 mẫu đầu làm validation.
- Tạo các biến train/validation tương ứng.

In [ ]:
# TODO: Tách validation set cho Reuters
x_val = x_train[:1000]
y_val = one_hot_train_labels[:1000]
partial_x_train = x_train[1000:]
partial_y_train = one_hot_train_labels[1000:]

# TODO: In shape các tập dữ liệu vừa tạo
print("x_val:", x_val.shape)
print("partial_x_train:", partial_x_train.shape)


## Bài 3.4 - Xây dựng mô hình Reuters

**Yêu cầu:**
- Tạo mô hình gồm:
  - Dense(64, relu)
  - Dense(64, relu)
  - Output layer: tự khai báo cho phù hợp
- Compile với:
  - optimizer = `rmsprop`
  - loss = `categorical_crossentropy`
  - metrics = `accuracy`

In [ ]:
# TODO: Tạo mô hình reuters_model
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10000,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(46, activation='softmax')])
# TODO: Compile mô hình
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# TODO: model.summary()
model.summary()


## Bài 3.5 - Train và quan sát overfitting

**Yêu cầu:**
- Train mô hình trong 20 epochs.
- batch_size = 256.
- validation_data = (x_val, y_val).
- Lưu vào `history`.

In [ ]:
# TODO: Train mô hình Reuters
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=256,
    validation_data=(x_val, y_val))
# history = ...


## Bài 3.6 - Vẽ biểu đồ loss / accuracy

**Yêu cầu:**
- Vẽ biểu đồ training loss và validation loss.
- Vẽ biểu đồ training accuracy và validation accuracy.
- Quan sát biểu đồ và cho biết: Nên chọn bao nhiêu epochs là hợp lý? Lý do?

In [ ]:
# TODO: Trích xuất loss, val_loss, accuracy, val_accuracy từ history.history
loss = history.history['loss']
val_loss = history.history['val_loss']
# TODO: Vẽ biểu đồ loss
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo-', label='Training loss')
plt.plot(epochs, val_loss, 'r^-', label='Validation loss')
plt.title('Training and Validation Loss (Reuters)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
# TODO: Vẽ biểu đồ accuracy
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plt.plot(epochs, acc, 'bo-', label='Training accuracy')
plt.plot(epochs, val_acc, 'r^-', label='Validation accuracy')
plt.title('Training and Validation Accuracy (Reuters)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Bài 3.7 - Huấn luyện mô hình cuối cùng

**Yêu cầu:**
- Tạo lại mô hình mới cùng kiến trúc.
- Huấn luyện với số epoch phù hợp hơn.
- Đánh giá trên test set.

In [ ]:
# TODO: Tạo reuters_final_model
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10000,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(46, activation='softmax')])
# TODO: Compile mô hình
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# TODO: Train lại trên toàn bộ x_train và one_hot_train_labels
model.fit(x_train, one_hot_train_labels,
          epochs=9,
          batch_size=256)


# TODO: Đánh giá trên test set
results = model.evaluate(x_test, one_hot_test_labels)

# TODO: In test_loss, test_accuracy
print("Test Loss:", results[0])
print("Test Accuracy:", results[1])


## Câu hỏi thảo luận - Reuters

1. Vì sao lớp cuối của mô hình dùng `Dense(46, activation='softmax')`?
Giúp mô hình chọn lớp có xác suất cao nhất
2. Vì sao nhãn cần one-hot hóa trong cách làm này?
softmax ở output
categorical_crossentropy làm loss
3. Nếu thay 46 bằng 10 ở lớp cuối thì có vấn đề gì?
Model chỉ dự đoán được 10 lớp thay vì 46
Trong khi dữ liệu có 46 lớp thật
4. Vì sao `categorical_crossentropy` phù hợp cho bài toán này?
Đây là bài toán phân loại đa lớp

# PHẦN 4 - BOSTON HOUSING: HỒI QUY

Trong phần này, bạn sẽ dự đoán giá nhà từ dữ liệu thuộc tính số.

## Bài 4.1 - Nạp dữ liệu Boston Housing

**Yêu cầu:**
- Nạp dữ liệu `boston_housing`.
- In số lượng mẫu train/test.
- In shape của dữ liệu và nhãn.

In [ ]:
# TODO: Nạp dữ liệu Boston Housing
(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()

# TODO: In shape train/test data và target
print("Số mẫu train:", len(train_data))
print("Số mẫu test:", len(test_data))
print("Shape train_data:", train_data.shape)
print("Shape train_targets:", train_targets.shape)
print("Shape test_data:", test_data.shape)
print("Shape test_targets:", test_targets.shape)

## Bài 4.2 - Chuẩn hóa dữ liệu

**Yêu cầu:**
- Tính `mean` theo từng cột từ `train_data`.
- Chuẩn hóa cả `train_data` và `test_data` bằng mean/std của train.
- In thử mean và std sau chuẩn hóa.

In [ ]:
# TODO: Tính mean theo từng cột từ training data
mean = train_data.mean(axis=0)

# TODO: Trừ mean cho train_data và test_data
train_data -= mean
test_data -= mean

# TODO: Tính std theo từng cột từ training data
std = train_data.std(axis=0)

# TODO: Chia train_data và test_data cho std
train_data /= std
test_data /= std

# TODO: In thử mean và std của vài cột đầu sau chuẩn hóa
print("Mean (5 cột đầu):", train_data.mean(axis=0)[:5])
print("Std (5 cột đầu):", train_data.std(axis=0)[:5])


## Bài 4.3 - Hàm tạo mô hình hồi quy

**Yêu cầu:**
- Viết hàm `build_model()` trả về mô hình gồm:
  - Dense(64, relu)
  - Dense(64, relu)
  - Output layer: tự khai báo cho phù hợp
- Compile với:
  - optimizer = `rmsprop`
  - loss = `mse`
  - metrics = `mae`

In [ ]:
def build_model():
    # TODO: Tạo mô hình hồi quy
        model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(train_data.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)])
    # TODO: Compile mô hình
    model.compile(optimizer='rmsprop',
       loss='mse',
       metrics=['mae'])
    return model

# TODO: Gọi thử build_model() và summary nếu cần
model = build_model()
model.summary()


## Bài 4.4 - K-fold validation

**Yêu cầu:**
- Chia dữ liệu train thành 5 fold.
- Mỗi lần lấy 1 fold làm validation, 4 fold còn lại để train.
- Train 100 epochs cho mỗi fold.
- Lưu MAE của từng fold vào `all_scores`.

In [ ]:
k = 5
num_val_samples = len(train_data) // k
num_epochs = 100
all_scores = []

for i in range(k):
    print('processing fold #', i)

    # TODO: Tạo validation data và validation targets
    val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
    val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]

    # TODO: Tạo partial training data và partial training targets
    partial_train_data = np.concatenate(
        [train_data[:i * num_val_samples],
         train_data[(i + 1) * num_val_samples:]],
        axis=0)
    partial_train_targets = partial_train_targets = np.concatenate(
        [train_targets[:i * num_val_samples],
         train_targets[(i + 1) * num_val_samples:]],
        axis=0)

    # TODO: Build model mới cho từng fold
    model = build_model()

    # TODO: Train model
        model.fit(partial_train_data, partial_train_targets,
              epochs=num_epochs,
              batch_size=16,
              verbose=0)

    # TODO: Đánh giá model trên val_data, val_targets
    val_mse, val_mae = model.evaluate(val_data, val_targets, verbose=0)
    all_scores.append(val_mae)

# TODO: In all_scores và MAE trung bình
print("MAE từng fold:", all_scores)
print("MAE trung bình:", np.mean(all_scores))


## Bài 4.5 - Theo dõi MAE theo epoch

**Yêu cầu:**
- Chạy lại K-fold validation nhưng lần này lưu toàn bộ lịch sử MAE theo từng epoch.
- Dùng 80 epochs để minh họa.
- Tính MAE trung bình theo epoch trên các fold.
- Vẽ đường cong `Validation MAE by Epoch`.
- Quan sát biểu đồ và cho biết: Nên chọn bao nhiêu epochs là hợp lý? Lý do?

In [ ]:
k = 4
num_val_samples = len(train_data) // k
num_epochs = 80
all_mae_histories = []

for i in range(k):
    print('processing fold #', i)

    # TODO: Tạo validation fold
    val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
    val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]

    # TODO: Tạo partial train
    partial_train_data = np.concatenate(
        [train_data[:i * num_val_samples],
         train_data[(i + 1) * num_val_samples:]],
        axis=0
    )

    partial_train_targets = partial_train_targets = np.concatenate(
        [train_targets[:i * num_val_samples],
         train_targets[(i + 1) * num_val_samples:]],
        axis=0
    )

    # TODO: Build model mới
    model = build_model()
    # TODO: Train và lưu history
  history = model.fit(
        partial_train_data,
        partial_train_targets,
        validation_data=(val_data, val_targets),
        epochs=num_epochs,
        batch_size=16,
        verbose=0
    )


    # TODO: Lấy mae history của fold hiện tại và append vào all_mae_histories
    mae_history = history.history['val_mae']
    all_mae_histories.append(mae_history)

# TODO: Tính average_mae_history
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories])
    for i in range(num_epochs)
]

# TODO: Vẽ average_mae_history
epochs = range(1, num_epochs + 1)

plt.plot(epochs, average_mae_history)
plt.xlabel('Epochs')
plt.ylabel('Validation MAE')
plt.title('Validation MAE by Epoch (Boston Housing)')
plt.show()


## Bài 4.6 - Huấn luyện mô hình cuối cùng và đánh giá trên test set

**Yêu cầu:**
- Tạo mô hình mới.
- Train với số epoch hợp lý dựa trên đường cong MAE vừa vẽ.
- Đánh giá trên test set.

In [ ]:
# TODO: Tạo final_model
model = build_model()

# TODO: Train final_model trên toàn bộ train_data, train_targets
model.fit(train_data, train_targets,
          epochs=25,
          batch_size=16,
          verbose=0)

# TODO: Đánh giá trên test_data, test_targets
test_mse, test_mae = model.evaluate(test_data, test_targets)

# TODO: In kết quả test_mse_score và test_mae_score
print("Test MSE:", test_mse)
print("Test MAE:", test_mae)


## Câu hỏi thảo luận - Boston Housing

1. Vì sao bài toán này không dùng `accuracy`?
accuracy chỉ dùng cho phân loại
2. Vì sao phải chuẩn hóa dữ liệu theo từng đặc trưng?
Các đặc trưng có thang giá trị khác nhau
3. Vì sao khi dữ liệu ít, K-fold validation đáng tin cậy hơn một lần train/validation split?
Kết quả phụ thuộc mạnh vào cách chia
4. Vì sao lớp cuối của mô hình hồi quy không dùng `sigmoid` hay `softmax`?
Vì đây là hồi quy dự đoán giá trị bất kỳ

# PHẦN 5 - TỔNG KẾT

## Bài tập tự đánh giá

Hoàn thành bảng sau vào vở hoặc markdown cell mới:

| Bài toán | Dạng đầu ra | Activation lớp cuối | Loss | Metric |
|---|---|---|---|---|
| IMDB | ? | ? | ? | ? |
| Iris | ? | ? | ? | ? |
| Reuters | ? | ? | ? | ? |
| Boston Housing | ? | ? | ? | ? |

## Câu hỏi vận dụng

1. Khi nào nên dùng `binary_crossentropy`?
2. Khi nào nên dùng `categorical_crossentropy`?
3. Khi nào nên dùng `mse` và `mae`?
4. Validation loss và training loss khác nhau thế nào về ý nghĩa?
5. Một mô hình có train accuracy rất cao nhưng val accuracy thấp cho biết điều gì?

